# Guía Completa sobre Conjuntos (`set`), Conjuntos Inmutables (`frozenset`) y Diccionarios Inmutables (`frozendict`) en Python

---

## 1. Introducción: ¿Qué son `set`, `frozenset` y `frozendict`?

En Python, los conjuntos y mapeos basados en **tablas hash** constituyen la base para trabajar con colecciones de elementos únicos y asociaciones clave-valor con tiempos de búsqueda constantes $O(1)$.

1. **`set` (Conjunto Mutable):** Colección **desordenada**, **mutable** y **sin elementos duplicados**. Implementa directamente la teoría matemática de conjuntos ($A \cup B, A \cap B, A \setminus B, A \triangle B$). Aunque el conjunto en sí es mutable (se pueden agregar y quitar elementos), **todos los elementos que contiene deben ser inmutables (*hashables*)**.
2. **`frozenset` (Conjunto Inmutable):** Es la contraparte **inmutable y *hashable*** de `set`. Una vez creado, no se pueden añadir ni eliminar elementos. Gracias a su inmutabilidad, un `frozenset` **sí puede usarse como clave de un diccionario (`dict`) o como elemento dentro de otro conjunto (`set`)**.
3. **`frozendict` (Diccionario Inmutable - PEP 814 / Python 3.15):** Nuevo tipo nativo (*built-in*) que representa un mapeo clave-valor **inmutable**. A diferencia de `dict` (que no es *hashable*) y de `types.MappingProxyType` (que es solo una vista envolvente sin soporte de `hash()`), `frozendict` bloquea cualquier mutación y **es *hashable*** siempre que todas sus claves y valores sean *hashables*.

### Tabla Comparativa General de Estructuras en Python

| Característica | `set` | `frozenset` | `dict` | `frozendict` (PEP 814) | `list` | `tuple` |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Sintaxis / Constructor** | `{1, 2}` o `set()` | `frozenset({1, 2})` | `{"a": 1}` | `frozendict({"a": 1})` | `[1, 2]` | `(1, 2)` |
| **Estructura de datos** | Elementos únicos | Elementos únicos | Pares `clave: valor` | Pares `clave: valor` | Secuencia indexada | Secuencia indexada |
| **Mutabilidad del contenedor** | **Mutable** | **Inmutable** | **Mutable** | **Inmutable** | **Mutable** | **Inmutable** |
| **¿Es *Hashable*?** | **No** | **Sí** | **No** | **Sí** (si los valores son *hashables*) | **No** | **Sí** (si sus elementos son *hashables*) |
| **Complejidad de búsqueda (`in`)** | **$O(1)$** promedio | **$O(1)$** promedio | **$O(1)$** promedio | **$O(1)$** promedio | $O(n)$ lineal | $O(n)$ lineal |
| **Orden de los elementos** | Desordenado | Desordenado | Orden de inserción | Orden de inserción | Posicional | Posicional |
| **¿Permite duplicados?** | **No** | **No** | Claves **No** / Valores Sí | Claves **No** / Valores Sí | Sí | Sí |

---
## 2. Creación y Estructura de `set` y `frozenset`

### ¡Regla de Oro con las Llaves Vacías `{}`!
- Por razones históricas de Python, escribir `{}` crea un **diccionario vacío (`dict`)**, **NUNCA** un conjunto vacío.
- Para crear un conjunto vacío mutable se debe usar obligatoriamente el constructor **`set()`**.
- Para crear un conjunto inmutable vacío se utiliza **`frozenset()`**.

In [ ]:
# 1. Creación de un set mediante sintaxis literal con llaves {}
numeros = {1, 2, 3, 4}
print("1. Literal de conjunto       :", numeros, "| Tipo:", type(numeros))

# 2. ¡Cuidado!: {} es un diccionario vacío, NO un conjunto vacío
no_es_set = {}
set_vacio = set()
print("2. Trampa de llaves vacías {}:", no_es_set, "| Tipo:", type(no_es_set))
print("   Conjunto vacío con set()  :", set_vacio, "| Tipo:", type(set_vacio), "| Longitud:", len(set_vacio))

# 3. Eliminación automática de duplicados a partir de una lista
lista_con_repetidos = [10, 20, 10, 30, 20, 10]
unicos = set(lista_con_repetidos)
print("3. Deduplicación automática  :", lista_con_repetidos, "->", unicos)

# 4. Creación a partir de cualquier iterable (por ejemplo, una cadena de texto)
letras = set("hola hola")
letras.discard(" ")
print("4. Desde un string (iterable):", letras, "(caracteres únicos sin orden fijo)")

# 5. Creación de un frozenset (no tiene sintaxis literal propia; usa siempre el constructor frozenset())
permisos_fijos = frozenset(["Lectura", "Escritura", "Ejecución", "Lectura"])
print("5. Conjunto inmutable        :", permisos_fijos, "| Tipo:", type(permisos_fijos))

# 6. frozenset vacío
fs_vacio = frozenset()
print("6. frozenset vacío           :", fs_vacio)

1. Literal de conjunto       : {1, 2, 3, 4} | Tipo: <class 'set'>
2. Trampa de llaves vacías {}: {} | Tipo: <class 'dict'>
   Conjunto vacío con set()  : set() | Tipo: <class 'set'> | Longitud: 0
3. Deduplicación automática  : [10, 20, 10, 30, 20, 10] -> {10, 20, 30}
4. Desde un string (iterable): {'o', 'h', 'l', 'a'} (caracteres únicos sin orden fijo)
5. Conjunto inmutable        : frozenset({'Lectura', 'Ejecución', 'Escritura'}) | Tipo: <class 'frozenset'>
6. frozenset vacío           : frozenset()


---
## 3. Arquitectura Interna en CPython: Tablas Hash, Mutabilidad y *Hashability*

### 3.1. ¿Cómo funciona internamente un `set` y un `frozenset`?
En CPython (`PySetObject`), un conjunto está implementado como una **Tabla Hash con direccionamiento abierto (*Open Addressing*)**, muy similar a un diccionario pero **sin punteros a valores** (solo almacena `(hash, key_ptr)`):

1. **Cálculo del Hash:** Al insertar o buscar un elemento `x`, Python calcula `hash(x)`.
2. **Salto Directo $O(1)$:** Mediante una máscara de bits (`hash(x) & mask`), salta directamente a la ranura (*slot*) correspondiente de la tabla en memoria.
3. **Sin Índice Posicional:** Como la posición de cada elemento depende de su valor hash y del tamaño actual de la tabla hash, **los conjuntos no tienen orden posicional** y no soportan indexación `s[0]` ni *slicing* `s[1:3]`.

### 3.2. ¿Por qué `set` es Mutable pero sus Elementos DEBEN ser Inmutables?
- Si pudieras meter una lista `[1, 2]` dentro de un `set` y luego mutarla a `[1, 2, 3]`, su valor hash cambiaría y el elemento quedaría **extraviado en una ranura incorrecta de la tabla hash**.
- Por eso, un `set` solo acepta elementos *hashables*: `int`, `float`, `str`, `bool`, `tuple` (de inmutables), **`frozenset`** y **`frozendict`**.
- En consecuencia, **un `set` NO puede contener otro `set`**, pero **un `set` SÍ puede contener objetos `frozenset`** (fundamental en matemáticas para construir conjuntos de conjuntos o el conjunto potencia $\mathcal{P}(A)$).

In [ ]:
# 1. Demostración de que 'set' es mutable in-place (conserva su dirección de memoria id())
s = {10, 20}
id_inicial = id(s)
print("ID de 'set' antes de mutar :", id_inicial)
s.add(30)
print(f"ID de 'set' después de mutar: {id(s)} (¿Mismo objeto en memoria?: {id(s) == id_inicial})")

# 2. Elementos válidos (hashables) vs inválidos (mutables) dentro de un set
print("\n--- Elementos Válidos e Inválidos dentro de un Set ---")
set_valido = {42, "Python", (1, 2), frozenset({10, 20})}
print("Set válido con tupla y frozenset:", set_valido)

try:
    {[1, 2, 3]}
except TypeError as e:
    print(f"Intentar meter una 'list' en un set -> {type(e).__name__}: {e}")

try:
    {{1, 2}}  # Un set dentro de otro set
except TypeError as e:
    print(f"Intentar meter un 'set' en un set   -> {type(e).__name__}: {e}")

# 3. Colisión de igualdad y hash: 1 == True == 1.0 y 0 == False == 0.0
print("\n--- Colisión de Igualdad: 1 == True == 1.0 y 0 == False == 0.0 ---")
colision = {1, True, 1.0, 0, False, 0.0}
print("Evaluación de {1, True, 1.0, 0, False, 0.0}:", colision, "(Se conserva el primero en aparecer)")

ID de 'set' antes de mutar : 140392819204160
ID de 'set' después de mutar: 140392819204160 (¿Mismo objeto en memoria?: True)

--- Elementos Válidos e Inválidos dentro de un Set ---
Set válido con tupla y frozenset: {(1, 2), 'Python', 42, frozenset({10, 20})}
Intentar meter una 'list' en un set -> TypeError: unhashable type: 'list'
Intentar meter un 'set' en un set   -> TypeError: unhashable type: 'set'

--- Colisión de Igualdad: 1 == True == 1.0 y 0 == False == 0.0 ---
Evaluación de {1, True, 1.0, 0, False, 0.0}: {0, 1} (Se conserva el primero en aparecer)


In [ ]:
import timeit

# Comparación empírica de velocidad de búsqueda (operador 'in')
n = 100_000
conjunto = set(range(n))
conjunto_fijo = frozenset(range(n))
lista = list(range(n))
objetivo = n - 1

t_set = timeit.timeit(lambda: objetivo in conjunto, number=1000)
t_frozenset = timeit.timeit(lambda: objetivo in conjunto_fijo, number=1000)
t_list = timeit.timeit(lambda: objetivo in lista, number=1000)

print("--- Benchmark de Pertenencia (in): O(1) vs O(n) sobre 100,000 elementos ---")
print(f"  Tiempo en set       O(1): {t_set:.5f}s")
print(f"  Tiempo en frozenset O(1): {t_frozenset:.5f}s")
print(f"  Tiempo en list      O(n): {t_list:.5f}s")

--- Benchmark de Pertenencia (in): O(1) vs O(n) sobre 100,000 elementos ---
  Tiempo en set       O(1): 0.00004s
  Tiempo en frozenset O(1): 0.00004s
  Tiempo en list      O(n): 0.91240s


---
## 4. Álgebra de Conjuntos: Operadores Simbólicos vs. Métodos

Tanto `set` como `frozenset` soportan las cuatro operaciones fundamentales del álgebra de conjuntos:

| Operación Matemática | Operador Simbólico | Método Equivalente | Definición Formal |
| :--- | :--- | :--- | :--- |
| **Unión** ($A \cup B$) | `A \| B` | `A.union(B, ...)` | Elementos que están en $A$, en $B$, o en ambos. |
| **Intersección** ($A \cap B$) | `A & B` | `A.intersection(B, ...)` | Elementos que están simultáneamente en $A$ y en $B$. |
| **Diferencia** ($A \setminus B$) | `A - B` | `A.difference(B, ...)` | Elementos que están en $A$ pero **no** están en $B$. |
| **Diferencia Simétrica** ($A \triangle B$) | `A ^ B` | `A.symmetric_difference(B)` | Elementos que están en $A$ o en $B$, pero **no en ambos** ($(A \cup B) \setminus (A \cap B)$). |

### Dos Reglas Técnicas Cruciales:
1. **Operadores vs. Métodos:** Los operadores simbólicos (`|`, `&`, `-`, `^`) **exigen que ambos operandos sean conjuntos (`set` o `frozenset`)**. En cambio, los métodos (`.union()`, `.intersection()`, etc.) **aceptan cualquier iterable** (listas, tuplas, generadores, rangos) y pueden encadenar múltiples argumentos.
2. **Tipo del Resultado al Mezclar `set` y `frozenset`:** Cuando se opera un `set` con un `frozenset`, **el tipo del objeto devuelto lo determina siempre el operando de la izquierda**.

In [ ]:
A = {1, 2, 3, 4}
B = {3, 4, 5, 6}

print("Conjunto A:", A)
print("Conjunto B:", B)

print("\n1. Unión (A | B)                :", A | B)
print("2. Intersección (A & B)         :", A & B)
print("3. Diferencia (A - B)           :", A - B)
print("4. Diferencia (B - A)           :", B - A)
print("5. Diferencia Simétrica (A ^ B) :", A ^ B)

# Diferencia entre operador | y método .union() con otros iterables
print("\n--- Ventaja de los Métodos: Aceptan cualquier iterable (ej. list y tuple) ---")
print("A.union([10, 20], (30, 40))     :", A.union([10, 20], (30, 40)))
try:
    _ = A | [10, 20]
except TypeError as e:
    print(f"Error con A | [10, 20]          : {type(e).__name__} -> {e}")

# Regla del operando izquierdo al mezclar set y frozenset
s_mut = {1, 2}
fs_inmut = frozenset({2, 3})
res1 = s_mut | fs_inmut
res2 = fs_inmut | s_mut
print("\n--- Regla del Operando Izquierdo (set vs frozenset) ---")
print(f"set | frozenset -> {res1} | Tipo: {type(res1)}")
print(f"frozenset | set -> {res2} | Tipo: {type(res2)}")

Conjunto A: {1, 2, 3, 4}
Conjunto B: {3, 4, 5, 6}

1. Unión (A | B)                : {1, 2, 3, 4, 5, 6}
2. Intersección (A & B)         : {3, 4}
3. Diferencia (A - B)           : {1, 2}
4. Diferencia (B - A)           : {5, 6}
5. Diferencia Simétrica (A ^ B) : {1, 2, 5, 6}

--- Ventaja de los Métodos: Aceptan cualquier iterable (ej. list y tuple) ---
A.union([10, 20], (30, 40))     : {1, 2, 3, 4, 10, 20, 30, 40}
Error con A | [10, 20]          : TypeError -> unsupported operand type(s) for |: 'set' and 'list'

--- Regla del Operando Izquierdo (set vs frozenset) ---
set | frozenset -> {1, 2, 3} | Tipo: <class 'set'>
frozenset | set -> frozenset({1, 2, 3}) | Tipo: <class 'frozenset'>


---
## 5. Relaciones de Inclusión, Comparación y Disjunción

En conjuntos, los operadores relacionales (`<=`, `<`, `>=`, `>`) **no evalúan orden alfabético ni numérico**, sino **relaciones de inclusión matemática**:

| Relación | Operador | Método | Significado Matemático |
| :--- | :--- | :--- | :--- |
| **Subconjunto** ($A \subseteq B$) | `A <= B` | `A.issubset(B)` | Todos los elementos de $A$ pertenecen a $B$. |
| **Subconjunto Propio** ($A \subsetneq B$) | `A < B` | *(Solo operador)* | $A \subseteq B$ y además $A \neq B$. |
| **Superconjunto** ($A \supseteq B$) | `A >= B` | `A.issuperset(B)` | $A$ contiene todos los elementos de $B$. |
| **Superconjunto Propio** ($A \supsetneq B$) | `A > B` | *(Solo operador)* | $A \supseteq B$ y además $A \neq B$. |
| **Conjuntos Disjuntos** ($A \cap B = \emptyset$) | *(Solo método)* | `A.isdisjoint(B)` | Retorna `True` si $A$ y $B$ **no tienen ningún elemento en común**. |
| **Igualdad Extensional** ($A = B$) | `A == B` | *(Solo operador)* | Poseen exactamente los mismos elementos, sin importar orden ni si son `set` o `frozenset`. |

In [ ]:
sub = {1, 2}
total = {1, 2, 3}
externo = {3, 4}

print("¿{1, 2} es subconjunto de {1, 2, 3}? (<=)       :", sub <= total)
print("¿{1, 2, 3} es subconjunto propio de {1, 2, 3}? (<):", total < total)
print("¿{1, 2, 3} es superconjunto de {1, 2}? (>=)     :", total >= sub)
print("¿{1, 2} y {3, 4} son disjuntos? (.isdisjoint)   :", sub.isdisjoint(externo))
print("¿{1, 2, 3} == frozenset([3, 1, 2])?             :", total == frozenset([3, 1, 2]))

¿{1, 2} es subconjunto de {1, 2, 3}? (<=)       : True
¿{1, 2, 3} es subconjunto propio de {1, 2, 3}? (<): False
¿{1, 2, 3} es superconjunto de {1, 2}? (>=)     : True
¿{1, 2} y {3, 4} son disjuntos? (.isdisjoint)   : True
¿{1, 2, 3} == frozenset([3, 1, 2])?             : True


---
## 6. Métodos de Mutación Exclusivos de `set` (vs. Inmutabilidad de `frozenset`)

Mientras que `frozenset` solo posee **8 métodos públicos** (todos de consulta o creación de nuevos conjuntos), `set` posee **17 métodos públicos**, añadiendo **9 métodos de mutación *in-place***:

### 6.1. Inserción y Eliminación Individual
- **`s.add(elem)`**: Añade un único elemento al conjunto.
- **`s.remove(elem)`**: Elimina `elem`. **Si no existe, lanza `KeyError`**.
- **`s.discard(elem)`**: Elimina `elem` si existe. **Si no existe, NO hace nada ni lanza error**.
- **`s.pop()`**: Elimina y devuelve un elemento arbitrario del conjunto (lanza `KeyError` si está vacío).
- **`s.clear()`**: Elimina todos los elementos dejando `set()`.

### 6.2. Mutación Algebraica *In-Place*
- **`s.update(it, ...)`** o **`s |= otro`**: Añade todos los elementos de uno o más iterables.
- **`s.intersection_update(it, ...)`** o **`s &= otro`**: Conserva solo los elementos comunes.
- **`s.difference_update(it, ...)`** o **`s -= otro`**: Elimina los elementos presentes en el otro conjunto.
- **`s.symmetric_difference_update(it)`** o **`s ^= otro`**: Actualiza `s` con la diferencia simétrica.

In [ ]:
metodos_fs = [m for m in dir(frozenset) if not m.startswith("_")]
metodos_exclusivos_set = sorted(set(dir(set)) - set(dir(frozenset)))

print(f"Métodos públicos de 'frozenset' ({len(metodos_fs)}):", metodos_fs)
print(f"Métodos exclusivos de 'set'     ({len(metodos_exclusivos_set)}):", metodos_exclusivos_set)

lenguajes = {"Python", "C++"}
lenguajes.add("SQL")
lenguajes.discard("Cobol")  # Seguro: no lanza excepción aunque 'Cobol' no exista

print("\n--- Prueba de .add(), .remove() vs .discard() ---")
print("Tras .add('SQL') y .discard('Cobol'):", lenguajes)

try:
    lenguajes.remove("Cobol")
except KeyError as e:
    print(f"Error al usar .remove('Cobol')       : {type(e).__name__} -> {e}")

# Actualización algebraica in-place
print("\n--- Actualización In-Place (|= y &=) conservando el mismo ID ---")
lenguajes |= {"Rust", "Go"}
print("Tras |= {'Rust', 'Go'}               :", lenguajes)
lenguajes &= {"Python", "Rust", "Java"}
print("Tras &= {'Python', 'Rust', 'Java'}   :", lenguajes)

Métodos públicos de 'frozenset' (8): ['copy', 'difference', 'intersection', 'isdisjoint', 'issubset', 'issuperset', 'symmetric_difference', 'union']
Métodos exclusivos de 'set'     (9): ['add', 'clear', 'difference_update', 'discard', 'intersection_update', 'pop', 'remove', 'symmetric_difference_update', 'update']

--- Prueba de .add(), .remove() vs .discard() ---
Tras .add('SQL') y .discard('Cobol'): {'Python', 'SQL', 'C++'}
Error al usar .remove('Cobol')       : KeyError -> 'Cobol'

--- Actualización In-Place (|= y &=) conservando el mismo ID ---
Tras |= {'Rust', 'Go'}               : {'Python', 'SQL', 'C++', 'Rust', 'Go'}
Tras &= {'Python', 'Rust', 'Java'}   : {'Python', 'Rust'}


---
## 7. Funciones Integradas (*Built-ins*) y Comprensión de Conjuntos (*Set Comprehensions*)

- **Funciones Integradas:** `len()`, `min()`, `max()`, `sum()`, `sorted()` (devuelve siempre una `list` ordenada), `any()` y `all()` funcionan directamente sobre `set` y `frozenset`.
- **Comprensión de Conjuntos (*Set Comprehension*):** Utiliza llaves `{}` sin dos puntos `:` (lo que la distingue de un *dict comprehension*):
  ```python
  {expresion for elemento in iterable if condicion}
  ```
- **Comprensión para `frozenset`:** Se pasa una expresión generadora al constructor `frozenset(expresion for elemento in iterable)`.

In [ ]:
muestras = {15, 4, 42, 8, 23, 16}
print("Conjunto de muestras:", muestras)
print(f"len(): {len(muestras)} | min(): {min(muestras)} | max(): {max(muestras)} | sum(): {sum(muestras)}")
print("sorted() devuelve lista ordenada:", sorted(muestras))

print("\n--- Set Comprehension y Frozenset por Comprensión ---")
# 1. Set comprehension: residuos cuadráticos módulo 7
residuos_mod7 = {(x ** 2) % 7 for x in range(20)}
print("Residuos cuadráticos mod 7 (set)      :", residuos_mod7)

# 2. Frozenset a partir de expresión generadora: extraer dominios únicos de correos
correos = ["ana@gmail.com", "danny@ucv.ve", "luis@gmail.com", "maria@ucv.ve"]
dominios = frozenset(correo.split("@")[1] for correo in correos)
print("Dominios de correo únicos (frozenset) :", dominios)

Conjunto de muestras: {4, 8, 15, 16, 23, 42}
len(): 6 | min(): 4 | max(): 42 | sum(): 108
sorted() devuelve lista ordenada: [4, 8, 15, 16, 23, 42]

--- Set Comprehension y Frozenset por Comprensión ---
Residuos cuadráticos mod 7 (set)      : {0, 1, 2, 4}
Dominios de correo únicos (frozenset) : frozenset({'ucv.ve', 'gmail.com'})


---
## 8. Apartado Especial: El Nuevo `frozendict` (PEP 814 / Python 3.15)

Históricamente, Python tenía parejas mutable/inmutable para secuencias (`list` / `tuple`) y para conjuntos (`set` / `frozenset`), pero carecía de un tipo nativo inmutable para diccionarios (`dict`). El **PEP 814** introduce **`frozendict`** como un tipo integrado (*built-in*) en Python 3.15.

### 8.1. ¿Por qué `frozendict` es superior a `types.MappingProxyType`?
Hasta ahora, cuando se quería exponer un diccionario de solo lectura se utilizaba `types.MappingProxyType(d)`. Sin embargo:
1. **`MappingProxyType` NO es verdaderamente inmutable:** Es solo una vista dinámica; si alguien conserva una referencia al `dict` subyacente `d` y lo modifica, el `MappingProxyType` cambia.
2. **`MappingProxyType` NO es *hashable*:** Llamar a `hash(MappingProxyType(d))` lanza `TypeError: unhashable type: 'mappingproxy'`. No puede usarse como clave de un `dict`, ni dentro de un `set`, ni en funciones decoradas con `@functools.lru_cache`.
3. **`frozendict` SÍ es *hashable* e inmutable de raíz:** Congela una copia propia en memoria (desacoplada del diccionario original), preserva el orden de inserción, mantiene búsqueda $O(1)$ y calcula su `hash()` en $O(n)$ la primera vez (cacheándolo) siempre que sus claves y valores sean *hashables*.

### 8.2. Comparativa Técnica: `dict` vs. `MappingProxyType` vs. `frozendict`

| Característica | `dict` | `types.MappingProxyType` | `frozendict` (PEP 814) |
| :--- | :--- | :--- | :--- |
| **¿Es un tipo *built-in*?** | Sí | No (módulo `types`) | **Sí** (en Python 3.15+) |
| **¿Permite mutar (`d[k] = v`)?** | Sí | No (`TypeError`) | **No (`TypeError`)** |
| **¿Aislado de mutaciones externas?** | No | **No** (cambia si muta el `dict` original) | **Sí** (inmutabilidad garantizada) |
| **¿Soporta `hash(d)`?** | **No** | **No** | **Sí** (si claves y valores son *hashables*) |
| **¿Puede ser clave de `dict` o elemento de `set`?** | **No** | **No** | **Sí** |
| **Operador de fusión `d1 \| d2`** | Sí (retorna `dict`) | Sí (retorna `dict`) | **Sí (retorna `frozendict`)** |
| **Métodos disponibles** | 11 métodos | `get`, `keys`, `values`, `items`, `copy` | `get`, `keys`, `values`, `items`, `copy`, `fromkeys` |

In [ ]:
import builtins
from collections.abc import Mapping

# Si estamos en Python 3.15+, usamos el built-in nativo 'frozendict' (PEP 814).
# En versiones anteriores (Python 3.10 - 3.14), definimos una implementación fiel al PEP 814
# para poder ejecutar todas las pruebas interactivamente en cualquier entorno.
if hasattr(builtins, "frozendict"):
    frozendict = builtins.frozendict
else:
    class frozendict(Mapping):
        """Implementación compatible con la especificación oficial del PEP 814."""
        __slots__ = ("_data", "_hash")

        def __init__(self, *args, **kwargs):
            self._data = dict(*args, **kwargs)
            self._hash = None

        def __getitem__(self, key):
            return self._data[key]

        def __iter__(self):
            return iter(self._data)

        def __len__(self):
            return len(self._data)

        def __repr__(self):
            return f"frozendict({self._data!r})"

        def __hash__(self):
            if self._hash is None:
                h = 0
                for k, v in self._data.items():
                    h ^= hash((k, v))
                self._hash = h
            return self._hash

        def __or__(self, other):
            if not isinstance(other, Mapping):
                return NotImplemented
            merged = dict(self._data)
            merged.update(other)
            return frozendict(merged)

        def copy(self):
            return self

        @classmethod
        def fromkeys(cls, iterable, value=None):
            return cls(dict.fromkeys(iterable, value))

# 1. Creación de un frozendict
hiperparametros = frozendict(modelo="Transformer", capas=12, lr=0.001)
print("1. Instancia de frozendict     :", hiperparametros)

# 2. Acceso por clave O(1) y vistas
print("2. Acceso O(1) y vistas        :", hiperparametros["modelo"], "| Claves:", list(hiperparametros.keys()))

# 3. Protección contra mutación
try:
    hiperparametros["lr"] = 0.01
except TypeError as e:
    print("3. Intento de mutación         : TypeError -> 'frozendict' object does not support item assignment")

# 4. Cálculo de hash y uso como clave de otro diccionario o elemento de un set
_ = hash(hiperparametros)
print("4. Hashabilidad (hash())       : Hash calculado exitosamente (entero de 64 bits)")

cache_experimentos = {hiperparametros: "Precisión 98.4%"}
print(f"5. Uso como clave en un dict   : Cache para {hiperparametros} -> {cache_experimentos[hiperparametros]}")

# 6. Operador de unión | (devuelve un nuevo frozendict sin alterar el original)
nuevos_params = hiperparametros | {"capas": 24, "lr": 0.0005, "dropout": 0.1}
print("6. Unión funcional con |       :", nuevos_params)

# 7. Hashabilidad condicional (igual que tuple): si un valor es mutable (list), no es hashable
fd_con_lista = frozendict({"capas": [64, 128]})
try:
    hash(fd_con_lista)
except TypeError as e:
    print(f"7. Con valor mutable (lista)   : Se puede crear, pero al pedir hash() -> {type(e).__name__}: {e}")

1. Instancia de frozendict     : frozendict({'modelo': 'Transformer', 'capas': 12, 'lr': 0.001})
2. Acceso O(1) y vistas        : Transformer | Claves: ['modelo', 'capas', 'lr']
3. Intento de mutación         : TypeError -> 'frozendict' object does not support item assignment
4. Hashabilidad (hash())       : Hash calculado exitosamente (entero de 64 bits)
5. Uso como clave en un dict   : Cache para frozendict({'modelo': 'Transformer', 'capas': 12, 'lr': 0.001}) -> Precisión 98.4%
6. Unión funcional con |       : frozendict({'modelo': 'Transformer', 'capas': 24, 'lr': 0.0005, 'dropout': 0.1})
7. Con valor mutable (lista)   : Se puede crear, pero al pedir hash() -> TypeError: unhashable type: 'list'


---
## 9. Resumen y Buenas Prácticas: ¿Cuándo usar cada estructura?

1. **Usa `set` cuando:**
   - Necesites **eliminar duplicados** rápidamente de una secuencia.
   - Realices verificaciones frecuentes de pertenencia (`x in coleccion`) en tiempo $O(1)$.
   - Necesites calcular intersecciones, uniones o diferencias entre grupos de datos dinámicos.
2. **Usa `frozenset` cuando:**
   - Necesites que un conjunto sea **clave de un diccionario** o **elemento dentro de otro conjunto** (por ejemplo, aristas `{u, v}` en grafos no dirigidos o estados en algoritmos de búsqueda).
   - Quieras definir constantes de conjuntos a nivel de módulo que ningún consumidor pueda alterar por accidente.
3. **Usa `frozendict` cuando:**
   - Manejes configuraciones globales, metadatos o hiperparámetros que deban permanecer estrictamente inmutables durante toda la ejecución.
   - Necesites almacenar diccionarios completos dentro de un `set` o usarlos como claves de caché en `@functools.lru_cache`.